# Fase 2: Transformación y Calidad de Datos (Transform)
En esta fase, tomaremos los datos extraídos y aplicaremos reglas de negocio para asegurar su calidad, integridad y consistencia antes de cargarlos al Data Warehouse:
- Eliminar registros duplicados.
- Estandarizar formatos de fecha.
- Corregir e imputar valores nulos o vacíos.
- Castear tipos de datos a su formato correspondiente (enteros, decimales, etc.).
- Estandarizar valores categóricos (ej. unificar canales de venta).

In [ ]:
import os
import json
import sqlite3
import pandas as pd


## 1. Extracción Inicial de los Datos Crudos
Cargamos los archivos de datos para aplicarles el proceso de transformación.

In [ ]:
DATA_DIR = "../data"
df_productos = pd.read_excel(os.path.join(DATA_DIR, "productos.xlsx"))
df_ventas = pd.read_csv(os.path.join(DATA_DIR, "ventas.csv"))
df_clientes = pd.read_json(os.path.join(DATA_DIR, "clientes.json"))

with sqlite3.connect(os.path.join(DATA_DIR, "inventario.db")) as conn:
    df_inventario_actual = pd.read_sql_query("SELECT * FROM inventario_actual;", conn)
    df_movimientos = pd.read_sql_query("SELECT * FROM movimientos_inventario;", conn)

with open(os.path.join(DATA_DIR, "api_marketing_response.json"), encoding="utf-8") as f:
    df_marketing = pd.json_normalize(json.load(f), record_path="campaigns")


## 2. Transformación de Clientes
- Eliminamos duplicados por `cliente_id` quedándonos con el último registro (`keep='last'`).
- Llenamos municipios y otros strings nulos o vacíos con "No disponible".
- Convertimos `fecha_registro` a formato `YYYY-MM-DD`.
- Casteamos `edad` a entero imputando nulos/inválidos con `0`.

In [ ]:
print(f"Clientes - Antes: {len(df_clientes)} filas")
df_cli_limpio = df_clientes.drop_duplicates(subset=["cliente_id"], keep="last").copy()
df_cli_limpio["municipio"] = df_cli_limpio["municipio"].replace("", "No disponible").fillna("No disponible")
df_cli_limpio["fecha_registro"] = pd.to_datetime(df_cli_limpio["fecha_registro"], errors="coerce").dt.strftime("%Y-%m-%d")

for col in df_cli_limpio.select_dtypes(include="object").columns:
    df_cli_limpio[col] = df_cli_limpio[col].fillna("No disponible")

df_cli_limpio["edad"] = pd.to_numeric(df_cli_limpio["edad"], errors="coerce").fillna(0).astype(int)
print(f"Clientes - Después: {len(df_cli_limpio)} filas")
df_cli_limpio.head(3)


## 3. Transformación de Productos
- Eliminamos duplicados por `producto_id`.
- Casteamos precios y costos a tipo numérico.
- Limpiamos espacios en blanco de strings y rellenamos nulos con "No disponible" o `0`.

In [ ]:
print(f"Productos - Antes: {len(df_productos)} filas")
df_prod_limpio = df_productos.drop_duplicates(subset=["producto_id"], keep="last").copy()
df_prod_limpio["costo_unitario"] = pd.to_numeric(df_prod_limpio["costo_unitario"], errors="coerce")
df_prod_limpio["precio_lista"] = pd.to_numeric(df_prod_limpio["precio_lista"], errors="coerce")

for col in df_prod_limpio.select_dtypes(include="object").columns:
    df_prod_limpio[col] = df_prod_limpio[col].astype(str).str.strip().replace("", "No disponible").fillna("No disponible")

for col in df_prod_limpio.select_dtypes(include="number").columns:
    df_prod_limpio[col] = df_prod_limpio[col].fillna(0)

print(f"Productos - Después: {len(df_prod_limpio)} filas")
df_prod_limpio.head(3)


## 4. Transformación de Ventas
- Eliminamos duplicados por `venta_id`.
- Parseamos fechas de diferentes formatos a un estándar `YYYY-MM-DD`.
- Estandarizamos canales de venta unificando `Web` como `E-commerce` y convirtiendo a Title Case.
- Casteamos `cantidad`, `precio_unitario`, `descuento` y `total_venta` a numéricos.
- Filtramos filas donde falten datos clave obligatorios (`venta_id`, `cliente_id`, `producto_id`, `fecha_venta`).

In [ ]:
def _parsear_fecha(valor):
    texto = str(valor).strip()
    if not texto or texto.lower() == "nan":
        return pd.NaT
    if "-" in texto and len(texto.split("-")[0]) == 4:
        return pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce")
    if "/" in texto:
        partes = texto.split("/")
        if len(partes[0]) == 4:
            return pd.to_datetime(texto, format="%Y/%m/%d", errors="coerce")
        return pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    return pd.to_datetime(texto, errors="coerce")

print(f"Ventas - Antes: {len(df_ventas)} filas")
df_ventas_limpio = df_ventas.drop_duplicates(subset=["venta_id"], keep="last").copy()
df_ventas_limpio["fecha_venta"] = df_ventas_limpio["fecha_venta"].apply(_parsear_fecha).dt.strftime("%Y-%m-%d")
df_ventas_limpio["canal_venta"] = df_ventas_limpio["canal_venta"].astype(str).str.strip().str.title().replace({"Web": "E-commerce"})
df_ventas_limpio["metodo_pago"] = df_ventas_limpio["metodo_pago"].fillna("No disponible")

for col in ["cantidad", "precio_unitario", "descuento", "total_venta"]:
    df_ventas_limpio[col] = pd.to_numeric(df_ventas_limpio[col], errors="coerce")

df_ventas_limpio = df_ventas_limpio.dropna(subset=["venta_id", "cliente_id", "producto_id", "fecha_venta"])
print(f"Ventas - Después: {len(df_ventas_limpio)} filas")
df_ventas_limpio.head(3)


## 5. Transformación de Inventario y Movimientos
- Eliminamos duplicados por `producto_id` + `bodega` (en existencias) y por `id` (en movimientos).
- Estandarizamos texto en bodegas y tipos de movimientos a Title Case.
- Casteamos cantidades a entero rellenando nulos con `0`.

In [ ]:
print(f"Existencias - Antes: {len(df_inventario_actual)} | Movimientos - Antes: {len(df_movimientos)}")
df_inv_limpio = df_inventario_actual.drop_duplicates(subset=["producto_id", "bodega"], keep="last").copy()
df_mov_limpio = df_movimientos.drop_duplicates(subset=["id"], keep="last").copy()

df_inv_limpio["bodega"] = df_inv_limpio["bodega"].astype(str).str.strip().str.title()
df_inv_limpio["existencia"] = pd.to_numeric(df_inv_limpio["existencia"], errors="coerce").fillna(0).astype(int)

df_mov_limpio["fecha"] = pd.to_datetime(df_mov_limpio["fecha"], errors="coerce").dt.strftime("%Y-%m-%d")
df_mov_limpio["tipo"] = df_mov_limpio["tipo"].astype(str).str.strip().str.title()
df_mov_limpio["cantidad"] = pd.to_numeric(df_mov_limpio["cantidad"], errors="coerce").fillna(0).astype(int)
df_mov_limpio = df_mov_limpio.dropna(subset=["fecha", "producto_id"])

print(f"Existencias - Después: {len(df_inv_limpio)} | Movimientos - Después: {len(df_mov_limpio)}")


## 6. Transformación de Marketing
- Renombramos columnas para evitar caracteres especiales.
- Eliminamos duplicados por ID de campaña.
- Casteamos campos numéricos e imputamos nulos con `0` o `0.0`.

In [ ]:
print(f"Marketing - Antes: {len(df_marketing)} filas")
df_mkt_limpio = df_marketing.copy()
# Manejar encoding alternativo si aplica
if "campaña_id" in df_mkt_limpio.columns:
    df_mkt_limpio = df_mkt_limpio.rename(columns={"campaña_id": "campana_id"})
if "campa\u00f1a_id" in df_mkt_limpio.columns:
    df_mkt_limpio = df_mkt_limpio.rename(columns={"campa\u00f1a_id": "campana_id"})

df_mkt_limpio = df_mkt_limpio.drop_duplicates(subset=["campana_id"], keep="last").copy()
df_mkt_limpio["fecha"] = pd.to_datetime(df_mkt_limpio["fecha"], errors="coerce").dt.strftime("%Y-%m-%d")
df_mkt_limpio["plataforma"] = df_mkt_limpio["plataforma"].astype(str).str.strip().str.title()

for col in ["impresiones", "clics", "leads", "conversiones"]:
    df_mkt_limpio[col] = pd.to_numeric(df_mkt_limpio[col], errors="coerce").fillna(0).astype(int)
df_mkt_limpio["costo"] = pd.to_numeric(df_mkt_limpio["costo"], errors="coerce").fillna(0.0)

print(f"Marketing - Después: {len(df_mkt_limpio)} filas")
df_mkt_limpio.head(3)
